In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load CSVs (they are in the current working directory)
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_df = pd.read_csv('error_logs.csv')

# Convert timestamp -> timezone-aware datetimes in UTC then to Asia/Shanghai
for df in (metric_df, trace_df, log_df, error_df):
    if 'timestamp' in df.columns:
        df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# 1) metric.csv aggregation
metric_grp = metric_df.groupby(['cmdb_id', 'kpi_name'], as_index=False).agg(
    count_points = ('value', 'count'),
    p95 = ('value', lambda s: float(s.quantile(0.95))),
    min_dt = ('dt', 'min'),
    max_dt = ('dt', 'max')
)
metric_grp['min_time'] = metric_grp['min_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
metric_grp['max_time'] = metric_grp['max_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
metric_agg = metric_grp[['cmdb_id','kpi_name','count_points','p95','min_time','max_time']].sort_values('count_points', ascending=False).head(50)

# 2) trace.csv aggregation
trace_grp = trace_df.groupby(['cmdb_id', 'trace_name'], as_index=False).agg(
    count_points = ('value', 'count'),
    p95 = ('value', lambda s: float(s.quantile(0.95))),
    min_dt = ('dt', 'min'),
    max_dt = ('dt', 'max')
)
trace_grp['min_time'] = trace_grp['min_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
trace_grp['max_time'] = trace_grp['max_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
trace_agg = trace_grp[['cmdb_id','trace_name','count_points','p95','min_time','max_time']].sort_values('count_points', ascending=False).head(50)

# 3) log.csv aggregation
log_grp = log_df.groupby(['cmdb_id', 'log_name'], as_index=False).agg(
    count_points = ('value', 'count'),
    p95 = ('value', lambda s: float(s.quantile(0.95))),
    min_dt = ('dt', 'min'),
    max_dt = ('dt', 'max')
)
log_grp['min_time'] = log_grp['min_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
log_grp['max_time'] = log_grp['max_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
log_agg = log_grp[['cmdb_id','log_name','count_points','p95','min_time','max_time']].sort_values('count_points', ascending=False).head(50)

# 4) error_logs.csv aggregation by cmdb_id (return all groups, limited to 20 rows for display)
err_grp = error_df.groupby(['cmdb_id'], as_index=False).agg(
    count_rows = ('message', 'count'),
    min_dt = ('dt', 'min'),
    max_dt = ('dt', 'max')
)
err_grp['min_time'] = err_grp['min_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
err_grp['max_time'] = err_grp['max_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
error_agg = err_grp[['cmdb_id','count_rows','min_time','max_time']].sort_values('count_rows', ascending=False).head(20)

# Display the compact aggregated tables (up to limits requested)
metric_agg, trace_agg, log_agg, error_agg

```
Out[1]:
```
```python
# Summarize the previously computed aggregates in plain English.
# Uses existing variables: metric_agg, trace_agg, log_agg, error_agg

def top_rows_summary(df, cols, n=3):
    rows = []
    for i in range(min(n, len(df))):
        r = df.iloc[i]
        parts = []
        for c in cols:
            parts.append(f"{c}={r[c]}")
        rows.append("; ".join(parts))
    return rows

parts = []

# Metric summary
m_count = metric_agg.shape[0]
parts.append(f"Metrics: {m_count} (grouped by cmdb_id,kpi_name) returned (top groups shown).")
if m_count > 0:
    m_top = top_rows_summary(metric_agg, ['cmdb_id','kpi_name','count_points','p95','min_time','max_time'], n=3)
    parts.append("Top metric groups (up to 3):")
    parts.extend([f" - {r}" for r in m_top])

# Trace summary
t_count = trace_agg.shape[0]
parts.append(f"Traces: {t_count} (grouped by cmdb_id,trace_name) returned (top groups shown).")
if t_count > 0:
    t_top = top_rows_summary(trace_agg, ['cmdb_id','trace_name','count_points','p95','min_time','max_time'], n=3)
    parts.append("Top trace groups (up to 3):")
    parts.extend([f" - {r}" for r in t_top])

# Log summary
l_count = log_agg.shape[0]
parts.append(f"Logs: {l_count} (grouped by cmdb_id,log_name) returned (top groups shown).")
if l_count > 0:
    l_top = top_rows_summary(log_agg, ['cmdb_id','log_name','count_points','p95','min_time','max_time'], n=5)
    parts.append("Top log groups (up to 5):")
    parts.extend([f" - {r}" for r in l_top])

# Error logs summary
e_count = error_agg.shape[0]
if e_count == 0:
    parts.append("Error logs: no rows/groups returned (empty).")
else:
    parts.append(f"Error logs: {e_count} groups returned (up to 20 shown).")
    e_top = top_rows_summary(error_agg, ['cmdb_id','count_rows','min_time','max_time'], n=5)
    parts.append("Top error_log groups (up to 5):")
    parts.extend([f" - {r}" for r in e_top])

summary = "\n".join(parts)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name  count_points        p95                 min_time                 max_time
412    Mysql01           Mysql-MySQL_3306_GetResponseTimeOfMysqld            31        0.0  2021-03-23 01:30:00 CST  2021-03-23 02:00:00 CST
1575  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...            31  1625363.5  2021-03-23 01:30:00 CST  2021-03-23 02:00:00 CST
411    Mysql01         Mysql-MySQL_3306_GetConnectedStateOfMysqld            31        1.0  2021-03-23 01:30:00 CST  2021-03-23 02:00:00 CST
1572  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...            31      138.0  2021-03-23 01:30:00 CST  2021-03-23 02:00:00 CST
1573  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...            31   230487.0  2021-03-23 01:30:00 CST  2021-03-23 02:00:00 CST
...        ...                                                ...           ...        ...                      ...                      ...
1338  Tomcat02     Tomcat-Sessions_7441--_SESSIONRejectedSessions            31        0.0  2021-03-23 01:30:00 CST  2021-03-23 02:00:00 CST
1458  Tomcat03     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter            31        0.0  2021-03-23 01:30:00 CST  2021-03-23 02:00:00 CST
1576  Tomcat04    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter            31        0.0  2021-03-23 01:30:00 CST  2021-03-23 02:00:00 CST
1577  Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...            31        0.0  2021-03-23 01:30:00 CST  2021-03-23 02:00:00 CST
1578  Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...            31        0.0  2021-03-23 01:30:00 CST  2021-03-23 02:00:00 CST

[50 rows x 6 columns],    cmdb_id                       trace_name  count_points         p95                 min_time                 max_time
0     IG01         trace.self.duration_mean            30    0.085747  2021-03-23 01:30:00 CST  2021-03-23 01:59:00 CST
1     IG01          trace.self.duration_p95            30    0.395847  2021-03-23 01:30:00 CST  2021-03-23 01:59:00 CST
2     IG01             trace.self.row_count            30  127.700000  2021-03-23 01:30:00 CST  2021-03-23 01:59:00 CST
6     IG01  trace.to_Tomcat02.duration_mean            30    0.120906  2021-03-23 01:30:00 CST  2021-03-23 01:59:00 CST
9     IG01  trace.to_Tomcat03.duration_mean            30    0.109665  2021-03-23 01:30:00 CST  2021-03-23 01:59:00 CST
..     ...                              ...           ...         ...                      ...                      ...
68    MG01      trace.to_dockerB2.row_count            30   16.100000  2021-03-23 01:30:00 CST  2021-03-23 01:59:00 CST
67    MG01   trace.to_dockerB2.duration_p95            30    0.483780  2021-03-23 01:30:00 CST  2021-03-23 01:59:00 CST
63    MG01  trace.to_dockerB1.duration_mean            30    0.117801  2021-03-23 01:30:00 CST  2021-03-23 01:59:00 CST
64    MG01   trace.to_dockerB1.duration_p95            30    0.368655  2021-03-23 01:30:00 CST  2021-03-23 01:59:00 CST
65    MG01      trace.to_dockerB1.row_count            30   19.100000  2021-03-23 01:30:00 CST  2021-03-23 01:59:00 CST

[50 rows x 6 columns],      cmdb_id          log_name  count_points     p95                 min_time                 max_time
11  Tomcat04     log.row_count            31  32.500  2021-03-23 01:30:00 CST  2021-03-23 02:00:00 CST
9   Tomcat04   log.error_count            31   0.000  2021-03-23 01:30:00 CST  2021-03-23 02:00:00 CST
0   Tomcat01   log.error_count            30   0.000  2021-03-23 01:30:00 CST  2021-03-23 01:59:00 CST
2   Tomcat01     log.row_count            30  31.000  2021-03-23 01:30:00 CST  2021-03-23 01:59:00 CST
6   Tomcat03   log.error_count            30   0.000  2021-03-23 01:30:00 CST  2021-03-23 01:59:00 CST
8   Tomcat03     log.row_count            30  31.200  2021-03-23 01:30:00 CST  2021-03-23 01:59:00 CST
3   Tomcat02   log.error_count            30   0.000  2021-03-23 01:30:00 CST  2021-03-23 01:59:00 CST
5   Tomcat02     log.row_count            30  31.100  2021-03-23 01:30:00 CST  2021-03-23 01:59:00 CST
14  apache02   log.error_count            30   0.000  2021-03-23 01:30:00 CST  2021-03-23 01:59:00 CST
13  apache01     log.row_count            30  62.700  2021-03-23 01:30:00 CST  2021-03-23 01:59:00 CST
15  apache02     log.row_count            30  64.400  2021-03-23 01:30:00 CST  2021-03-23 01:59:00 CST
12  apache01   log.error_count            30   0.000  2021-03-23 01:30:00 CST  2021-03-23 01:59:00 CST
10  Tomcat04  log.gc_pause_max             3   0.079  2021-03-23 01:32:00 CST  2021-03-23 02:00:00 CST
7   Tomcat03  log.gc_pause_max             1   0.050  2021-03-23 01:30:00 CST  2021-03-23 01:30:00 CST
1   Tomcat01  log.gc_pause_max             1   0.120  2021-03-23 01:52:00 CST  2021-03-23 01:52:00 CST
4   Tomcat02  log.gc_pause_max             1   0.030  2021-03-23 01:49:00 CST  2021-03-23 01:49:00 CST, Empty DataFrame
Columns: [cmdb_id, count_rows, min_time, max_time]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

# Return top 20 component-KPI groups sorted by global P95 (descending).
# Reuse metric_agg if available, otherwise re-aggregate metric.csv.

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

try:
    metric_agg  # use existing aggregation if present
except NameError:
    # load and aggregate metric.csv
    metric_df = pd.read_csv('metric.csv')
    metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    metric_grp = metric_df.groupby(['cmdb_id', 'kpi_name'], as_index=False).agg(
        count_points = ('value', 'count'),
        p95 = ('value', lambda s: float(s.quantile(0.95))),
        min_dt = ('dt', 'min'),
        max_dt = ('dt', 'max')
    )
    metric_grp['min_time'] = metric_grp['min_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    metric_grp['max_time'] = metric_grp['max_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    metric_agg = metric_grp[['cmdb_id','kpi_name','count_points','p95','min_time','max_time']]

top20_by_p95 = metric_agg.sort_values('p95', ascending=False).head(20)[['cmdb_id','kpi_name','count_points','p95','min_time','max_time']]
top20_by_p95

```
Out[2]:
```
```python
# Produce a concise plain-English summary of the top 20 component-KPI groups by global P95.
try:
    df = top20_by_p95
except NameError:
    import pandas as pd, pytz
    tz = pytz.timezone('Asia/Shanghai')
    metric_df = pd.read_csv('metric.csv')
    metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    metric_grp = metric_df.groupby(['cmdb_id', 'kpi_name'], as_index=False).agg(
        count_points = ('value', 'count'),
        p95 = ('value', lambda s: float(s.quantile(0.95))),
        min_dt = ('dt', 'min'),
        max_dt = ('dt', 'max')
    )
    metric_grp['min_time'] = metric_grp['min_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    metric_grp['max_time'] = metric_grp['max_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    df = metric_grp[['cmdb_id','kpi_name','count_points','p95','min_time','max_time']].sort_values('p95', ascending=False).head(20)

# Build summary
total = len(df)
global_start = df['min_time'].min() if total>0 else ''
global_end = df['max_time'].max() if total>0 else ''
top_n = min(5, total)
lines = [f"Top {total} component-KPI groups by global P95 (descending).",
         f"Time window covered (per-group ranges): earliest min_time = {global_start}, latest max_time = {global_end}.",
         f"Top {top_n} entries by P95:"]
for i, row in df.head(top_n).iterrows():
    p95_str = format(float(row['p95']), '.6g')
    lines.append(f"{len(lines)-2}) {row['cmdb_id']} - {row['kpi_name']}: p95={p95_str}, count={int(row['count_points'])}, range={row['min_time']} to {row['max_time']}")

summary = " ".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  count_points           p95                 min_time                 max_time
284       MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax            31  2.040136e+09  2021-03-23 01:30:00 CST  2021-03-23 02:00:00 CST
286       MG02          JVM-Memory_7779_JVM_Memory_HeapMemoryUsed            31  9.505183e+08  2021-03-23 01:30:00 CST  2021-03-23 02:00:00 CST
1574  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_Processin...            31  8.531126e+08  2021-03-23 01:30:00 CST  2021-03-23 02:00:00 CST
287       MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed            31  5.786694e+07  2021-03-23 01:30:00 CST  2021-03-23 02:00:00 CST
283       MG01                 OSLinux-OSLinux_ZABBIX_Host_Uptime            31  7.635630e+06  2021-03-23 01:30:00 CST  2021-03-23 02:00:00 CST
1029   Redis02                 OSLinux-OSLinux_ZABBIX_Host_Uptime            31  6.085892e+06  2021-03-23 01:30:00 CST  2021-03-23 02:00:00 CST
1575  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...            31  1.625364e+06  2021-03-23 01:30:00 CST  2021-03-23 02:00:00 CST
1573  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...            31  2.304870e+05  2021-03-23 01:30:00 CST  2021-03-23 02:00:00 CST
1468  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...            31  5.000000e+02  2021-03-23 01:30:00 CST  2021-03-23 02:00:00 CST
1572  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...            31  1.380000e+02  2021-03-23 01:30:00 CST  2021-03-23 02:00:00 CST
1466  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...            31  1.000000e+02  2021-03-23 01:30:00 CST  2021-03-23 02:00:00 CST
963    Redis02                        OSLinux-CPU_CPU_CPUidleutil            31  9.960708e+01  2021-03-23 01:30:00 CST  2021-03-23 02:00:00 CST
290       MG02           JVM-Threads_7779_JVM_ThreadCount_Threads            31  7.500000e+01  2021-03-23 01:30:00 CST  2021-03-23 02:00:00 CST
304       MG02                        OSLinux-CPU_CPU_CPUidleutil            31  7.447642e+01  2021-03-23 01:30:00 CST  2021-03-23 02:00:00 CST
302       MG02                        OSLinux-CPU_CPU_CPUUserTime            31  2.520931e+01  2021-03-23 01:30:00 CST  2021-03-23 02:00:00 CST
1022   Redis02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount            31  4.000000e+00  2021-03-23 01:30:00 CST  2021-03-23 02:00:00 CST
276       MG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount            31  3.000000e+00  2021-03-23 01:30:00 CST  2021-03-23 02:00:00 CST
272       MG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies            31  2.000000e+00  2021-03-23 01:30:00 CST  2021-03-23 02:00:00 CST
300       MG02                            OSLinux-CPU_CPU_CPULoad            31  1.075000e+00  2021-03-23 01:30:00 CST  2021-03-23 02:00:00 CST
411    Mysql01         Mysql-MySQL_3306_GetConnectedStateOfMysqld            31  1.000000e+00  2021-03-23 01:30:00 CST  2021-03-23 02:00:00 CST```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure metric_df and metric_agg / top20_by_p95 exist; otherwise load/recompute
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
    metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

try:
    metric_agg
except NameError:
    # compute global p95 per group (full series)
    metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    metric_grp = metric_df.groupby(['cmdb_id', 'kpi_name'], as_index=False).agg(
        count_points=('value','count'),
        p95=('value', lambda s: float(s.quantile(0.95))),
        min_dt=('dt','min'),
        max_dt=('dt','max')
    )
    metric_grp['min_time'] = metric_grp['min_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    metric_grp['max_time'] = metric_grp['max_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    metric_agg = metric_grp[['cmdb_id','kpi_name','count_points','p95','min_time','max_time']]

try:
    top20_by_p95
except NameError:
    top20_by_p95 = metric_agg.sort_values('p95', ascending=False).head(20)

# 1) Use the top 20 groups by global P95 from prior ranking
top20 = top20_by_p95[['cmdb_id','kpi_name','p95']].copy()

# 2) Filter metric.csv to the time window (inclusive) in Asia/Shanghai (UTC+8)
start_local = tz.localize(pd.to_datetime('2021-03-23 01:30:00'))
end_local = tz.localize(pd.to_datetime('2021-03-23 02:00:00'))

# ensure metric_df.dt exists and is timezone-aware Asia/Shanghai
if 'dt' not in metric_df.columns:
    metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure timezone
    if metric_df['dt'].dt.tz is None:
        metric_df['dt'] = metric_df['dt'].dt.tz_localize('UTC').dt.tz_convert(tz)
    else:
        metric_df['dt'] = metric_df['dt'].dt.tz_convert(tz)

window_mask = (metric_df['dt'] >= start_local) & (metric_df['dt'] <= end_local)
window_df = metric_df[window_mask].copy()

# 3) For each selected group, compute required aggregates using global p95
# Merge global p95 into window rows for comparison
window_top = window_df.merge(top20, on=['cmdb_id','kpi_name'], how='inner')

# flag anomalies (value > global p95)
window_top['is_anomaly'] = window_top['value'] > window_top['p95']

# group-level aggregates from windowed data
agg_window = window_top.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
    count_points_in_window=('value','count'),
    anomaly_count=('is_anomaly','sum'),
    max_value_in_window=('value','max')
)

# earliest anomaly per group (if any)
anom_rows = window_top[window_top['is_anomaly']].sort_values(['cmdb_id','kpi_name','dt'])
if not anom_rows.empty:
    earliest_anom = anom_rows.groupby(['cmdb_id','kpi_name'], as_index=False).first()[['cmdb_id','kpi_name','dt','value']]
    earliest_anom = earliest_anom.rename(columns={'dt':'earliest_anomaly_time','value':'earliest_anomaly_value'})
else:
    earliest_anom = pd.DataFrame(columns=['cmdb_id','kpi_name','earliest_anomaly_time','earliest_anomaly_value'])

# Combine results: start from top20 to include groups with zero rows in window
result = top20.merge(agg_window, on=['cmdb_id','kpi_name'], how='left')
result = result.merge(earliest_anom, on=['cmdb_id','kpi_name'], how='left')

# Fill missing window counts and anomaly counts with zeros
result['count_points_in_window'] = result['count_points_in_window'].fillna(0).astype(int)
result['anomaly_count'] = result['anomaly_count'].fillna(0).astype(int)
# max_value_in_window may be NaN if no rows in window
# earliest_anomaly_time/value remain NaT/NaN for missing

# Format earliest_anomaly_time to readable string, keep NULLs as None
def fmt_dt(x):
    if pd.isna(x):
        return None
    return x.tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

result['earliest_anomaly_time'] = result['earliest_anomaly_time'].apply(fmt_dt)
# earliest_anomaly_value keep as float or None
result['earliest_anomaly_value'] = result['earliest_anomaly_value'].where(~pd.isna(result['earliest_anomaly_value']), None)
# max_value_in_window keep numeric or NaN

# Reorder columns as requested
out_cols = ['cmdb_id','kpi_name','count_points_in_window','p95','anomaly_count','earliest_anomaly_time','earliest_anomaly_value','max_value_in_window']
final = result[out_cols].copy()

# 4) Sort: groups with anomaly_count>0 first ordered by earliest_anomaly_time ascending, then anomaly_count==0
final['has_anom'] = final['anomaly_count'] > 0
# For sorting, replace None with a large timestamp so groups without anomaly go to the end when sorting by earliest_anomaly_time
# But we will sort by has_anom desc (True first) then earliest_anomaly_time asc (None stay last)
final = final.sort_values(by=['has_anom','earliest_anomaly_time'], ascending=[False, True])

# Drop helper column
final = final.drop(columns=['has_anom'])

# Limit to 20 rows (top20 anyway), keep compact
final.head(20)

```
Out[3]:
```
```python
# Summarize the windowed anomaly results in plain English using the existing `final` DataFrame.
# Display a concise summary string.

import pandas as pd

# use existing 'final' DataFrame from previous step
df = final.copy()

total_groups = len(df)
groups_with_anom = df[df['anomaly_count'] > 0].copy()
n_groups_with_anom = len(groups_with_anom)
n_groups_no_anom = total_groups - n_groups_with_anom

# Prepare up to 5 example anomaly lines sorted by earliest anomaly time
examples = []
if n_groups_with_anom > 0:
    groups_with_anom_sorted = groups_with_anom.sort_values('earliest_anomaly_time', na_position='last')
    for _, r in groups_with_anom_sorted.head(5).iterrows():
        examples.append(
            f"{r['cmdb_id']} | {r['kpi_name']} — anomalies={int(r['anomaly_count'])}, "
            f"earliest={r['earliest_anomaly_time']}, value={float(r['earliest_anomaly_value']):.6g}, "
            f"p95={float(r['p95']):.6g}, max_in_window={float(r['max_value_in_window']):.6g}"
        )

summary_lines = [
    f"Top-20 groups evaluated (by global P95) within window 2021-03-23 01:30:00 to 02:00:00 (UTC+8):",
    f"- Total groups checked: {total_groups}",
    f"- Groups with >=1 point above global P95 in the window: {n_groups_with_anom}",
    f"- Groups with no points above global P95 in the window: {n_groups_no_anom}",
]

if examples:
    summary_lines.append("- Up to 5 earliest anomaly examples:")
    for ex in examples:
        summary_lines.append(f"  • {ex}")

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  count_points_in_window           p95  anomaly_count    earliest_anomaly_time  earliest_anomaly_value  max_value_in_window
18      MG02                            OSLinux-CPU_CPU_CPULoad                      31  1.075000e+00              2  2021-03-23 01:30:00 CST            1.080000e+00         1.090000e+00
14      MG02                        OSLinux-CPU_CPU_CPUUserTime                      31  2.520931e+01              2  2021-03-23 01:32:00 CST            2.526080e+01         2.526080e+01
1       MG02          JVM-Memory_7779_JVM_Memory_HeapMemoryUsed                      31  9.505183e+08              2  2021-03-23 01:39:00 CST            9.516837e+08         9.787158e+08
13      MG02                        OSLinux-CPU_CPU_CPUidleutil                      31  7.447642e+01              2  2021-03-23 01:44:00 CST            7.447750e+01         7.452838e+01
11   Redis02                        OSLinux-CPU_CPU_CPUidleutil                      31  9.960708e+01              2  2021-03-23 01:49:00 CST            9.961332e+01         9.961352e+01
2   Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_Processin...                      31  8.531126e+08              2  2021-03-23 01:59:00 CST            8.531128e+08         8.531131e+08
4       MG01                 OSLinux-OSLinux_ZABBIX_Host_Uptime                      31  7.635630e+06              2  2021-03-23 01:59:00 CST            7.635660e+06         7.635719e+06
5    Redis02                 OSLinux-OSLinux_ZABBIX_Host_Uptime                      31  6.085892e+06              2  2021-03-23 01:59:00 CST            6.085923e+06         6.085983e+06
6   Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...                      31  1.625364e+06              2  2021-03-23 01:59:00 CST            1.625371e+06         1.625380e+06
0       MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax                      31  2.040136e+09              0                     None                     NaN         2.040136e+09
3       MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed                      31  5.786694e+07              0                     None                     NaN         5.786694e+07
7   Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...                      31  2.304870e+05              0                     None                     NaN         2.304870e+05
8   Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...                      31  5.000000e+02              0                     None                     NaN         5.000000e+02
9   Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...                      31  1.380000e+02              0                     None                     NaN         1.380000e+02
10  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...                      31  1.000000e+02              0                     None                     NaN         1.000000e+02
12      MG02           JVM-Threads_7779_JVM_ThreadCount_Threads                      31  7.500000e+01              0                     None                     NaN         7.500000e+01
15   Redis02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount                      31  4.000000e+00              0                     None                     NaN         4.000000e+00
16      MG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount                      31  3.000000e+00              0                     None                     NaN         3.000000e+00
17      MG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies                      31  2.000000e+00              0                     None                     NaN         2.000000e+00
19   Mysql01         Mysql-MySQL_3306_GetConnectedStateOfMysqld                      31  1.000000e+00              0                     None                     NaN         1.000000e+00```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse metric_df, metric_agg, and final if available; otherwise load/compute as needed
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
    metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# ensure dt is present and in Asia/Shanghai
if 'dt' not in metric_df.columns:
    metric_df['dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    metric_df['dt'] = pd.to_datetime(metric_df['dt'], utc=True).dt.tz_convert(tz)

try:
    metric_agg
except NameError:
    metric_grp = metric_df.groupby(['cmdb_id','kpi_name'], as_index=False).agg(
        p95 = ('value', lambda s: float(s.quantile(0.95)))
    )
    metric_agg = metric_grp

# Recreate the window mask for March 23, 2021 01:30:00 through 02:00:00 (inclusive) UTC+8
start_local = tz.localize(pd.to_datetime('2021-03-23 01:30:00'))
end_local = tz.localize(pd.to_datetime('2021-03-23 02:00:00'))
window_mask = (metric_df['dt'] >= start_local) & (metric_df['dt'] <= end_local)
window_df = metric_df[window_mask].copy()

# Identify groups that had anomaly_count>0 from previous scan (use 'final' if available)
try:
    groups_with_anom = final[final['anomaly_count'] > 0][['cmdb_id','kpi_name','p95']].drop_duplicates().reset_index(drop=True)
except NameError:
    # fallback: compute groups where any point in window > global p95
    merged = window_df.merge(metric_agg[['cmdb_id','kpi_name','p95']], on=['cmdb_id','kpi_name'], how='left')
    merged['is_anom'] = merged['value'] > merged['p95']
    groups_with_anom = merged[merged['is_anom']].groupby(['cmdb_id','kpi_name'], as_index=False).agg(p95=('p95','first'))

# Prepare container for results
rows = []

for _, gr in groups_with_anom.iterrows():
    cmdb = gr['cmdb_id']
    kpi = gr['kpi_name']
    p95 = float(gr['p95'])
    grp = window_df[(window_df['cmdb_id'] == cmdb) & (window_df['kpi_name'] == kpi)].sort_values('dt').reset_index(drop=True)
    count_points_in_window = int(grp.shape[0])
    if count_points_in_window == 0:
        # no data in window (unlikely for groups previously flagged), produce zeros/NULLs
        rows.append({
            'cmdb_id': cmdb,
            'kpi_name': kpi,
            'p95': p95,
            'count_points_in_window': 0,
            'total_anomaly_points': 0,
            'num_anomaly_runs': 0,
            'longest_run_length': 0,
            'earliest_run_start_time': None,
            'earliest_run_max_value': None,
            'earliest_run_max_over_p95': None,
            'max_value_in_window': None
        })
        continue

    values = grp['value'].astype(float).to_numpy()
    dts = grp['dt'].to_numpy()

    # boolean anomaly mask (value > global p95)
    anomaly = values > p95
    total_anomaly_points = int(np.sum(anomaly))

    # compute runs using run-length logic
    if total_anomaly_points == 0:
        num_runs = 0
        longest_run = 0
        earliest_start = None
        earliest_max_val = None
        earliest_max_over_p95 = None
    else:
        # identify boundaries where anomaly state changes
        diff = np.concatenate(([True], anomaly[1:] != anomaly[:-1], [True]))  # not directly used, simpler approach below
        # run ids
        run_id = (anomaly != np.roll(anomaly, 1)).cumsum()
        # But the above run_id flawed for first element; better:
        run_change = np.empty_like(anomaly, dtype=bool)
        run_change[0] = True
        run_change[1:] = anomaly[1:] != anomaly[:-1]
        run_id = np.cumsum(run_change)

        run_df = pd.DataFrame({
            'run_id': run_id,
            'is_anom': anomaly,
            'value': values,
            'dt': dts
        })

        # group by run_id and keep only anomaly runs
        runs = run_df.groupby('run_id').agg(
            is_anom=('is_anom','first'),
            start_dt=('dt','first'),
            length=('is_anom','size'),
            max_value=('value','max')
        ).reset_index()

        anom_runs = runs[runs['is_anom'] == True].copy()
        num_runs = int(anom_runs.shape[0])
        if num_runs == 0:
            longest_run = 0
            earliest_start = None
            earliest_max_val = None
            earliest_max_over_p95 = None
        else:
            longest_run = int(anom_runs['length'].max())
            # earliest run start time
            earliest_row = anom_runs.sort_values('start_dt').iloc[0]
            earliest_start = earliest_row['start_dt']
            earliest_max_val = float(earliest_row['max_value'])
            earliest_max_over_p95 = earliest_max_val / p95 if p95 != 0 else None

    max_value_in_window = float(np.nanmax(values)) if values.size>0 else None

    # format earliest_start to readable string or None
    if earliest_start is None or pd.isna(earliest_start):
        earliest_start_str = None
    else:
        # earliest_start is numpy datetime64[ns, tz], convert via pandas Timestamp
        earliest_start_str = pd.to_datetime(earliest_start).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

    rows.append({
        'cmdb_id': cmdb,
        'kpi_name': kpi,
        'p95': p95,
        'count_points_in_window': count_points_in_window,
        'total_anomaly_points': total_anomaly_points,
        'num_anomaly_runs': num_runs,
        'longest_run_length': longest_run,
        'earliest_run_start_time': earliest_start_str,
        'earliest_run_max_value': earliest_max_val if total_anomaly_points>0 else None,
        'earliest_run_max_over_p95': earliest_max_over_p95 if total_anomaly_points>0 else None,
        'max_value_in_window': max_value_in_window
    })

# Build DataFrame
anom_runs_df = pd.DataFrame(rows, columns=[
    'cmdb_id','kpi_name','p95','count_points_in_window','total_anomaly_points',
    'num_anomaly_runs','longest_run_length','earliest_run_start_time',
    'earliest_run_max_value','earliest_run_max_over_p95','max_value_in_window'
])

# Sorting: rows with num_anomaly_runs>0 first ordered by earliest_run_start_time ascending, then others
# Convert earliest_run_start_time to datetime for sorting, keep NaT for None
anom_runs_df['_earliest_dt'] = pd.to_datetime(anom_runs_df['earliest_run_start_time']).dt.tz_localize(tz, ambiguous='NaT', nonexistent='NaT')
anom_runs_df = anom_runs_df.sort_values(by=['num_anomaly_runs', '_earliest_dt'], ascending=[False, True])
anom_runs_df = anom_runs_df.drop(columns=['_earliest_dt'])

# Keep output compact: only requested columns, limited to top 20 rows
anom_runs_df.head(20)

```
Out[4]:
```
```python
# Build a concise plain-English summary from the previously computed `anom_runs_df`.
# Reuse in-memory DataFrame and present a compact summary string.

import pandas as pd

df = anom_runs_df.head(20).copy()  # reuse result from prior step (limited to 20)

lines = []
lines.append(f"Analyzed {len(df)} component-KPI groups (window 2021-03-23 01:30–02:00 UTC+8).")
lines.append(f"Groups with anomaly runs (value > global P95): {int((df['num_anomaly_runs']>0).sum())}.\n")

for i, r in df.iterrows():
    idx = i + 1
    cmdb = r['cmdb_id']
    kpi = r['kpi_name']
    p95 = float(r['p95'])
    total_anom = int(r['total_anomaly_points'])
    runs = int(r['num_anomaly_runs'])
    longest = int(r['longest_run_length'])
    start = r['earliest_run_start_time'] or "N/A"
    earliest_max = r['earliest_run_max_value']
    ratio = r['earliest_run_max_over_p95']
    max_win = r['max_value_in_window']

    lines.append(
        f"{idx}. {cmdb} - {kpi} | p95={p95:.6g} | anomalies={total_anom} in {runs} run(s) | "
        f"longest_run={longest} min | earliest_run_start={start} | "
        f"earliest_run_max={earliest_max:.6g} ({ratio:.6g}x p95) | max_in_window={max_win:.6g}"
    )

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name           p95  count_points_in_window  total_anomaly_points  num_anomaly_runs  longest_run_length  earliest_run_start_time  earliest_run_max_value  earliest_run_max_over_p95  max_value_in_window
0      MG02                            OSLinux-CPU_CPU_CPULoad  1.075000e+00                      31                     2                 2                   1  2021-03-23 01:30:00 CST            1.080000e+00                   1.004651         1.090000e+00
1      MG02                        OSLinux-CPU_CPU_CPUUserTime  2.520931e+01                      31                     2                 2                   1  2021-03-23 01:32:00 CST            2.526080e+01                   1.002042         2.526080e+01
2      MG02          JVM-Memory_7779_JVM_Memory_HeapMemoryUsed  9.505183e+08                      31                     2                 2                   1  2021-03-23 01:39:00 CST            9.516837e+08                   1.001226         9.787158e+08
3      MG02                        OSLinux-CPU_CPU_CPUidleutil  7.447642e+01                      31                     2                 2                   1  2021-03-23 01:44:00 CST            7.447750e+01                   1.000015         7.452838e+01
4   Redis02                        OSLinux-CPU_CPU_CPUidleutil  9.960708e+01                      31                     2                 2                   1  2021-03-23 01:49:00 CST            9.961332e+01                   1.000063         9.961352e+01
5  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_Processin...  8.531126e+08                      31                     2                 1                   2  2021-03-23 01:59:00 CST            8.531131e+08                   1.000001         8.531131e+08
6      MG01                 OSLinux-OSLinux_ZABBIX_Host_Uptime  7.635630e+06                      31                     2                 1                   2  2021-03-23 01:59:00 CST            7.635719e+06                   1.000012         7.635719e+06
7   Redis02                 OSLinux-OSLinux_ZABBIX_Host_Uptime  6.085892e+06                      31                     2                 1                   2  2021-03-23 01:59:00 CST            6.085983e+06                   1.000015         6.085983e+06
8  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...  1.625364e+06                      31                     2                 1                   2  2021-03-23 01:59:00 CST            1.625380e+06                   1.000010         1.625380e+06```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure dataframes are loaded (reuse if present)
try:
    log_df
except NameError:
    log_df = pd.read_csv('log.csv')
try:
    error_df
except NameError:
    error_df = pd.read_csv('error_logs.csv')
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

# Ensure dt columns exist and are timezone-aware Asia/Shanghai
def ensure_dt(df):
    if 'dt' in df.columns:
        # coerce to datetime then set tz to UTC if naive, then convert
        df['dt'] = pd.to_datetime(df['dt'])
        if df['dt'].dt.tz is None:
            # assume original numeric timestamp available to reconstruct if possible
            if 'timestamp' in df.columns:
                df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
            else:
                df['dt'] = df['dt'].dt.tz_localize('UTC').dt.tz_convert(tz)
        else:
            df['dt'] = df['dt'].dt.tz_convert(tz)
    else:
        # create from numeric timestamp
        if 'timestamp' in df.columns:
            df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        else:
            raise ValueError("DataFrame lacks 'timestamp' column to create dt")

for df in (log_df, error_df, trace_df):
    ensure_dt(df)

# Window boundaries (inclusive) in UTC+8
start_local = tz.localize(pd.to_datetime('2021-03-23 01:30:00'))
end_local = tz.localize(pd.to_datetime('2021-03-23 02:00:00'))

# Components and log_names of interest
components = ['Tomcat04','Tomcat01','Tomcat02','Tomcat03','MG02','MG01','Redis02','Mysql01']
log_names = ['log.error_count','log.row_count','log.gc_pause_max']

# 1) Filter log.csv
log_mask = (
    log_df['cmdb_id'].isin(components) &
    log_df['log_name'].isin(log_names) &
    (log_df['dt'] >= start_local) & (log_df['dt'] <= end_local)
)
log_sel = log_df.loc[log_mask, ['dt','cmdb_id','log_name','value']].copy()
log_sel['timestamp'] = log_sel['dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
log_out = log_sel[['timestamp','cmdb_id','log_name','value']].sort_values('timestamp', ascending=True).head(50)

# 2) Filter error_logs.csv
err_mask = (
    error_df['cmdb_id'].isin(components) &
    (error_df['dt'] >= start_local) & (error_df['dt'] <= end_local)
)
err_sel = error_df.loc[err_mask, ['dt','cmdb_id','message']].copy()
err_sel['timestamp'] = err_sel['dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
error_out = err_sel[['timestamp','cmdb_id','message']].sort_values('timestamp', ascending=True).head(50)

# 3) Trace filtering and grouping
trace_cmps = ['IG01','MG01','MG02','Tomcat04','Tomcat03','Tomcat02','Tomcat01']
# filter trace rows by window, component, and trace_name contains keywords
name_mask = trace_df['trace_name'].str.contains('Tomcat|to_Tomcat|trace.self', regex=True, na=False)
trace_mask = (
    trace_df['cmdb_id'].isin(trace_cmps) &
    name_mask &
    (trace_df['dt'] >= start_local) & (trace_df['dt'] <= end_local)
)
trace_win = trace_df.loc[trace_mask, ['cmdb_id','trace_name','dt','value']].copy()

# compute full-series global P95 per (cmdb_id, trace_name) BEFORE filtering
trace_p95_full = trace_df.groupby(['cmdb_id','trace_name'], as_index=False).agg(
    p95 = ('value', lambda s: float(s.quantile(0.95)))
)

# aggregate counts and min/max times from the windowed data
if not trace_win.empty:
    trace_grouped = trace_win.groupby(['cmdb_id','trace_name'], as_index=False).agg(
        count_points_in_window = ('value','count'),
        min_time_dt = ('dt','min'),
        max_time_dt = ('dt','max')
    )
    trace_grouped['min_time'] = trace_grouped['min_time_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    trace_grouped['max_time'] = trace_grouped['max_time_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    # merge with global p95
    trace_merged = trace_grouped.merge(trace_p95_full, on=['cmdb_id','trace_name'], how='left')
    trace_out = trace_merged[['cmdb_id','trace_name','count_points_in_window','min_time','max_time','p95']].sort_values('count_points_in_window', ascending=False).head(50)
else:
    trace_out = pd.DataFrame(columns=['cmdb_id','trace_name','count_points_in_window','min_time','max_time','p95'])

# Return the three compact outputs (use variable names, not print)
log_out, error_out, trace_out

```
Out[5]:
```
```python
# Build a concise plain-English summary from the previously produced results:
# log_out, error_out, trace_out are expected to exist in memory.

# Compose summary
log_count = int(log_out.shape[0])
err_count = int(error_out.shape[0])
trace_groups = int(trace_out.shape[0])

# Prepare small examples (up to 3) for logs and trace
def row_examples(df, cols, n=3):
    ex = []
    for _, r in df.head(n).iterrows():
        parts = []
        for c in cols:
            parts.append(f"{c}={r[c]}")
        ex.append(", ".join(parts))
    return ex

log_examples = row_examples(log_out, ['timestamp','cmdb_id','log_name','value'], n=3)
trace_examples = row_examples(trace_out, ['cmdb_id','trace_name','count_points_in_window','min_time','max_time','p95'], n=3)

lines = []
lines.append(f"1) Logs (2021-03-23 01:30–02:00 UTC+8): {log_count} rows matched for components "
             f"['Tomcat04','Tomcat01','Tomcat02','Tomcat03','MG02','MG01','Redis02','Mysql01'] "
             f"and log_names ['log.error_count','log.row_count','log.gc_pause_max'].")
if log_examples:
    lines.append("   Example rows (first 3):")
    for e in log_examples:
        lines.append(f"    • {e}")

lines.append(f"2) Error logs (same window & components): {err_count} rows found.")
if err_count == 0:
    lines.append("   No error_log rows in the selected window for the given components.")

lines.append(f"3) Traces (filtered by names containing 'Tomcat'/'to_Tomcat'/'trace.self' and components "
             f"['IG01','MG01','MG02','Tomcat04','Tomcat03','Tomcat02','Tomcat01']): {trace_groups} groups returned.")
if trace_examples:
    lines.append("   Top trace groups (by count in window, first 3):")
    for e in trace_examples:
        lines.append(f"    • {e}")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                  timestamp   cmdb_id         log_name  value
0   2021-03-23 01:30:00 CST  Tomcat01  log.error_count    0.0
1   2021-03-23 01:30:00 CST  Tomcat01    log.row_count   23.0
2   2021-03-23 01:30:00 CST  Tomcat02  log.error_count    0.0
3   2021-03-23 01:30:00 CST  Tomcat02    log.row_count   29.0
4   2021-03-23 01:30:00 CST  Tomcat03  log.error_count    0.0
..                      ...       ...              ...    ...
67  2021-03-23 01:35:00 CST  Tomcat03    log.row_count   21.0
63  2021-03-23 01:35:00 CST  Tomcat01    log.row_count   23.0
64  2021-03-23 01:35:00 CST  Tomcat02  log.error_count    0.0
62  2021-03-23 01:35:00 CST  Tomcat01  log.error_count    0.0
66  2021-03-23 01:35:00 CST  Tomcat03  log.error_count    0.0

[50 rows x 4 columns], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [],      cmdb_id                       trace_name  count_points_in_window                 min_time                 max_time         p95
0       IG01         trace.self.duration_mean                      30  2021-03-23 01:30:00 CST  2021-03-23 01:59:00 CST    0.085747
1       IG01          trace.self.duration_p95                      30  2021-03-23 01:30:00 CST  2021-03-23 01:59:00 CST    0.395847
2       IG01             trace.self.row_count                      30  2021-03-23 01:30:00 CST  2021-03-23 01:59:00 CST  127.700000
7       IG01   trace.to_Tomcat02.duration_p95                      30  2021-03-23 01:30:00 CST  2021-03-23 01:59:00 CST    0.493480
6       IG01  trace.to_Tomcat02.duration_mean                      30  2021-03-23 01:30:00 CST  2021-03-23 01:59:00 CST    0.120906
..       ...                              ...                     ...                      ...                      ...         ...
45  Tomcat01         trace.self.duration_mean                      30  2021-03-23 01:30:00 CST  2021-03-23 01:59:00 CST    0.019274
46  Tomcat01          trace.self.duration_p95                      30  2021-03-23 01:30:00 CST  2021-03-23 01:59:00 CST    0.066767
47  Tomcat01             trace.self.row_count                      30  2021-03-23 01:30:00 CST  2021-03-23 01:59:00 CST  210.550000
48  Tomcat02         trace.self.duration_mean                      30  2021-03-23 01:30:00 CST  2021-03-23 01:59:00 CST    0.015684
56  Tomcat04             trace.self.row_count                      30  2021-03-23 01:30:00 CST  2021-03-23 01:59:00 CST  219.800000

[50 rows x 6 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load trace.csv if not already loaded
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

# Ensure dt column exists and is timezone-aware Asia/Shanghai
if 'dt' in trace_df.columns:
    trace_df['dt'] = pd.to_datetime(trace_df['dt'])
    if trace_df['dt'].dt.tz is None:
        if 'timestamp' in trace_df.columns:
            trace_df['dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        else:
            trace_df['dt'] = trace_df['dt'].dt.tz_localize('UTC').dt.tz_convert(tz)
    else:
        trace_df['dt'] = trace_df['dt'].dt.tz_convert(tz)
else:
    trace_df['dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Selection criteria
components = ['IG01','Tomcat04','Tomcat03','Tomcat02','Tomcat01']
name_pattern = r'trace\.self|to_Tomcat|to_Tomcat04'

# Compute full-series global P95 per (cmdb_id, trace_name)
trace_global_p95 = trace_df.groupby(['cmdb_id','trace_name'], as_index=False).agg(
    global_p95 = ('value', lambda s: float(s.quantile(0.95)))
)

# Select groups matching component and name pattern
selected_groups = trace_global_p95[
    trace_global_p95['cmdb_id'].isin(components) &
    trace_global_p95['trace_name'].str.contains(name_pattern, regex=True, na=False)
].copy()

# Window boundaries (inclusive) in UTC+8
start_local = tz.localize(pd.to_datetime('2021-03-23 01:30:00'))
end_local = tz.localize(pd.to_datetime('2021-03-23 02:00:00'))

# Prepare windowed trace rows for those components and names
window_mask = (
    (trace_df['dt'] >= start_local) & (trace_df['dt'] <= end_local) &
    trace_df['cmdb_id'].isin(components) &
    trace_df['trace_name'].str.contains(name_pattern, regex=True, na=False)
)
trace_window = trace_df[window_mask].copy()

# Build results by iterating selected groups to ensure using global_p95 from full series
rows = []
for _, row in selected_groups.iterrows():
    cmdb = row['cmdb_id']
    trace_name = row['trace_name']
    global_p95 = float(row['global_p95'])
    grp = trace_window[(trace_window['cmdb_id']==cmdb) & (trace_window['trace_name']==trace_name)].sort_values('dt')
    count_points_in_window = int(grp.shape[0])
    if count_points_in_window == 0:
        anomaly_count = 0
        earliest_anom_time = None
        earliest_anom_value = None
        max_value_in_window = None
    else:
        vals = grp['value'].astype(float)
        max_value_in_window = float(vals.max()) if not vals.empty else None
        anom_mask = vals > global_p95
        anomaly_count = int(anom_mask.sum())
        if anomaly_count == 0:
            earliest_anom_time = None
            earliest_anom_value = None
        else:
            anom_rows = grp[anom_mask]
            earliest_row = anom_rows.nsmallest(1, 'dt').iloc[0]
            earliest_anom_time = earliest_row['dt'].tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
            earliest_anom_value = float(earliest_row['value'])

    rows.append({
        'cmdb_id': cmdb,
        'trace_name': trace_name,
        'count_points_in_window': count_points_in_window,
        'global_p95': global_p95,
        'anomaly_count': anomaly_count,
        'earliest_anomaly_time': earliest_anom_time,
        'earliest_anomaly_value': earliest_anom_value,
        'max_value_in_window': max_value_in_window
    })

out = pd.DataFrame(rows, columns=[
    'cmdb_id','trace_name','count_points_in_window','global_p95',
    'anomaly_count','earliest_anomaly_time','earliest_anomaly_value','max_value_in_window'
])

# Sort: anomaly_count>0 first ordered by earliest_anomaly_time ascending, then groups with anomaly_count==0
# Convert earliest_anomaly_time to datetime for sorting (NaT for None)
out['_earliest_dt'] = pd.to_datetime(out['earliest_anomaly_time']).dt.tz_localize(tz, ambiguous='NaT', nonexistent='NaT')
out['has_anom'] = out['anomaly_count'] > 0
out = out.sort_values(by=['has_anom','_earliest_dt'], ascending=[False, True]).drop(columns=['_earliest_dt','has_anom'])

# Limit to 30 rows and keep compact
out.head(30)

```
Out[6]:
```
```python
# Targeted trace anomaly analysis with sustained-run detection and localization hints.
import pandas as pd
import pytz
from io import StringIO

tz = pytz.timezone('Asia/Shanghai')

# Load trace.csv
trace_df = pd.read_csv('trace.csv')

# Ensure dt column with UTC+8 tz
trace_df['dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Selection criteria
components = ['IG01','Tomcat04','Tomcat03','Tomcat02','Tomcat01']
name_pattern = r'trace\.self|to_Tomcat|to_Tomcat04'

# Compute global P95 per group (full series)
trace_global_p95 = trace_df.groupby(['cmdb_id','trace_name'], as_index=False).agg(
    global_p95 = ('value', lambda s: float(s.quantile(0.95)))
)

# Select relevant groups
selected = trace_global_p95[
    trace_global_p95['cmdb_id'].isin(components) &
    trace_global_p95['trace_name'].str.contains(name_pattern, regex=True, na=False)
].copy()

# Window boundaries (inclusive)
start_local = tz.localize(pd.to_datetime('2021-03-23 01:30:00'))
end_local   = tz.localize(pd.to_datetime('2021-03-23 02:00:00'))

# Windowed trace rows matching components and names
mask_window = (
    (trace_df['dt'] >= start_local) & (trace_df['dt'] <= end_local) &
    trace_df['cmdb_id'].isin(components) &
    trace_df['trace_name'].str.contains(name_pattern, regex=True, na=False)
)
trace_win = trace_df[mask_window].copy()

# Helper: compute run info for anomaly timestamps (per-minute series)
def compute_runs(times):
    if len(times) == 0:
        return 0, 0, None  # num_runs, longest_run_len, earliest_run_start (Timestamp)
    times = pd.to_datetime(times).sort_values().drop_duplicates()
    diffs = times.to_series().diff().dt.total_seconds().fillna(9999)
    # start of a new run where diff > 60 (gap) -> new run
    run_id = (diffs > 60).cumsum()
    runs = pd.DataFrame({'time': times.values, 'run_id': run_id.values})
    grouped = runs.groupby('run_id').agg(start_time=('time','first'), length=('time','size'))
    grouped = grouped.reset_index(drop=True)
    num_runs = int(grouped.shape[0])
    longest = int(grouped['length'].max())
    earliest_start = pd.to_datetime(grouped.sort_values('start_time').iloc[0]['start_time'])
    return num_runs, longest, earliest_start

# Build analysis rows
rows = []
# Also collect earliest self anomalies per Tomcat to help localization
tomcat_self_earliest = {}

for _, r in selected.iterrows():
    cmdb = r['cmdb_id']
    name = r['trace_name']
    global_p95 = float(r['global_p95'])
    grp = trace_win[(trace_win['cmdb_id']==cmdb) & (trace_win['trace_name']==name)].sort_values('dt')
    count_points = int(grp.shape[0])
    if count_points == 0:
        anomaly_count = 0
        earliest_anom_time = None
        earliest_anom_value = None
        max_in_window = None
        num_runs = 0
        longest_run = 0
        earliest_run_start = None
    else:
        vals = grp['value'].astype(float)
        max_in_window = float(vals.max())
        anom_mask = vals > global_p95
        anomaly_count = int(anom_mask.sum())
        if anomaly_count == 0:
            earliest_anom_time = None
            earliest_anom_value = None
            num_runs = 0
            longest_run = 0
            earliest_run_start = None
        else:
            anom_rows = grp[anom_mask]
            earliest_row = anom_rows.nsmallest(1, 'dt').iloc[0]
            earliest_anom_time = earliest_row['dt'].tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
            earliest_anom_value = float(earliest_row['value'])
            # compute runs
            times = anom_rows['dt']
            num_runs, longest_run, earliest_run_start_ts = compute_runs(times)
            earliest_run_start = None if earliest_run_start_ts is None else earliest_run_start_ts.tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

    rows.append({
        'cmdb_id': cmdb,
        'trace_name': name,
        'count_points_in_window': count_points,
        'global_p95': global_p95,
        'anomaly_count': anomaly_count,
        'earliest_anomaly_time': earliest_anom_time,
        'earliest_anomaly_value': earliest_anom_value,
        'max_value_in_window': max_in_window,
        'num_anomaly_runs': num_runs,
        'longest_run_length_min': longest_run,
        'earliest_run_start_time': earliest_run_start
    })

df_out = pd.DataFrame(rows)

# Determine Tomcat self earliest anomaly times for localization (if any)
self_mask = df_out['trace_name'].str.contains(r'trace\.self', regex=True, na=False) & df_out['cmdb_id'].str.contains('Tomcat')
for _, row in df_out[self_mask].iterrows():
    key = row['cmdb_id']
    t = row['earliest_anomaly_time']
    if pd.notna(t) and t is not None:
        tomcat_self_earliest[key] = pd.to_datetime(t).tz_localize(tz)

# For traces that are to_Tomcat*, check if their earliest anomaly precedes the corresponding Tomcat self anomaly
localization_notes = []
for _, r in df_out.iterrows():
    name = r['trace_name']
    if 'to_Tomcat' in name:
        # try to parse target Tomcat from trace name (e.g., to_Tomcat02)
        import re
        m = re.search(r'to_Tomcat0?\d+', name)
        if m:
            target = m.group(0).replace('to_','')  # e.g., Tomcat02
            et = r['earliest_anomaly_time']
            if et is not None:
                et_dt = pd.to_datetime(et).tz_localize(tz)
                tomcat_et = tomcat_self_earliest.get(target)
                if tomcat_et is None:
                    localization_notes.append(f"{r['cmdb_id']} -> {target}: trace anomalies start at {et} (no early Tomcat self anomaly recorded).")
                else:
                    if et_dt < tomcat_et:
                        localization_notes.append(f"{r['cmdb_id']} -> {target}: trace anomalies start at {et}, earlier than {target} self anomaly at {tomcat_et.strftime('%Y-%m-%d %H:%M:%S %Z')}, suggesting latency observed upstream into {target}.")
                    else:
                        localization_notes.append(f"{r['cmdb_id']} -> {target}: trace anomalies start at {et}, after {target} self anomaly at {tomcat_et.strftime('%Y-%m-%d %H:%M:%S %Z')}.")
            else:
                localization_notes.append(f"{r['cmdb_id']} -> {target}: no anomalies in window.")

# Sorting: anomaly_count>0 first ordered by earliest_anomaly_time ascending
df_out['_et'] = pd.to_datetime(df_out['earliest_anomaly_time']).dt.tz_localize(tz, ambiguous='NaT', nonexistent='NaT')
df_out['has_anom'] = df_out['anomaly_count'] > 0
df_out = df_out.sort_values(by=['has_anom','_et'], ascending=[False, True]).drop(columns=['_et','has_anom'])

# Limit to 30 rows, but create concise summary string highlighting sustained runs (>1 min) and localization hints
sustained = df_out[df_out['longest_run_length_min'] > 1]
sustained_list = []
for _, r in sustained.iterrows():
    sustained_list.append(f"{r['cmdb_id']} | {r['trace_name']} — earliest={r['earliest_anomaly_time']}, longest_run={int(r['longest_run_length_min'])} min")

anoms = df_out[df_out['anomaly_count']>0]
summary_lines = []
summary_lines.append("Trace anomaly summary (window 2021-03-23 01:30–02:00 UTC+8):")
summary_lines.append(f"- Selected groups checked: {len(df_out)}")
summary_lines.append(f"- Groups with anomalies in window: {int((df_out['anomaly_count']>0).sum())}")
if len(sustained_list)>0:
    summary_lines.append(f"- Groups with sustained runs (>1 min): {len(sustained_list)}")
    for s in sustained_list:
        summary_lines.append(f"  • {s}")
else:
    summary_lines.append("- No sustained runs (>1 minute) detected among selected groups.")

# Show earliest anomaly examples (up to 6)
if not anoms.empty:
    summary_lines.append("- Earliest anomaly examples (first 6 by time):")
    ex = anoms.sort_values('earliest_anomaly_time').head(6)
    for _, r in ex.iterrows():
        summary_lines.append(
            f"  • {r['cmdb_id']} | {r['trace_name']} — earliest={r['earliest_anomaly_time']}, count_in_window={r['count_points_in_window']}, anomaly_count={r['anomaly_count']}, max_in_window={r['max_value_in_window']}"
        )

# Localization hints from to_Tomcat traces
if localization_notes:
    summary_lines.append("- Localization hints from to_Tomcat traces:")
    for note in localization_notes:
        summary_lines.append(f"  • {note}")
else:
    summary_lines.append("- No localization hints from to_Tomcat traces.")

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                       trace_name  count_points_in_window  global_p95  anomaly_count    earliest_anomaly_time  earliest_anomaly_value  max_value_in_window
13      IG01   trace.to_Tomcat04.duration_p95                      30    0.196890              2  2021-03-23 01:30:00 CST                0.202200             0.423900
2       IG01             trace.self.row_count                      30  127.700000              2  2021-03-23 01:31:00 CST              168.000000           168.000000
5       IG01      trace.to_Tomcat01.row_count                      29   16.200000              2  2021-03-23 01:31:00 CST               30.000000            30.000000
8       IG01      trace.to_Tomcat02.row_count                      30   19.650000              2  2021-03-23 01:31:00 CST               21.000000            25.000000
14      IG01      trace.to_Tomcat04.row_count                      30   16.650000              2  2021-03-23 01:31:00 CST               18.000000            20.000000
..       ...                              ...                     ...         ...            ...                      ...                     ...                  ...
19  Tomcat02          trace.self.duration_p95                      30    0.074792              2  2021-03-23 01:45:00 CST                0.385650             0.385650
25  Tomcat04          trace.self.duration_p95                      30    0.035875              2  2021-03-23 01:45:00 CST                0.037750             0.037750
9       IG01  trace.to_Tomcat03.duration_mean                      30    0.109665              2  2021-03-23 01:54:00 CST                0.694000             0.694000
21  Tomcat03         trace.self.duration_mean                      30    0.019844              2  2021-03-23 01:54:00 CST                0.038062             0.038062
11      IG01      trace.to_Tomcat03.row_count                      30   17.000000              0                     None                     NaN            17.000000

[27 rows x 8 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse trace_df if available, otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

# Ensure dt column exists and is timezone-aware (Asia/Shanghai)
if 'dt' in trace_df.columns:
    trace_df['dt'] = pd.to_datetime(trace_df['dt'])
    if trace_df['dt'].dt.tz is None:
        if 'timestamp' in trace_df.columns:
            trace_df['dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        else:
            trace_df['dt'] = trace_df['dt'].dt.tz_localize('UTC').dt.tz_convert(tz)
    else:
        trace_df['dt'] = trace_df['dt'].dt.tz_convert(tz)
else:
    trace_df['dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Define target groups and window (inclusive)
targets = [
    ('IG01','trace.to_Tomcat04.duration_p95'),
    ('IG01','trace.to_Tomcat04.row_count'),
    ('IG01','trace.self.row_count'),
    ('Tomcat04','trace.self.duration_p95'),
    ('Tomcat04','trace.self.row_count')
]
target_df = pd.DataFrame(targets, columns=['cmdb_id','trace_name'])

start_local = tz.localize(pd.to_datetime('2021-03-23 01:30:00'))
end_local   = tz.localize(pd.to_datetime('2021-03-23 02:00:00'))

# Filter by window and exact group matches
mask_window = (trace_df['dt'] >= start_local) & (trace_df['dt'] <= end_local)
mask_group = trace_df[['cmdb_id','trace_name']].merge(target_df, on=['cmdb_id','trace_name'], how='inner').index
# More straightforward: use isin on tuple pairs
mask_group2 = trace_df.apply(lambda r: (r['cmdb_id'], r['trace_name']) in set(targets), axis=1)

filtered = trace_df[mask_window & mask_group2].copy()

# Format timestamp readable in UTC+8
filtered['timestamp'] = filtered['dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# Select requested columns and sort by timestamp ascending
selected_rows = filtered[['timestamp','cmdb_id','trace_name','value']].sort_values('timestamp', ascending=True).head(200)

# Display the result (variable, not print)
selected_rows

```
Out[7]:
```
```python
# Summarize the filtered trace rows succinctly as a plain-English string.
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

try:
    selected_rows
except NameError:
    # fallback: reload and filter if selected_rows not in memory
    trace_df = pd.read_csv('trace.csv')
    trace_df['dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    targets = [
        ('IG01','trace.to_Tomcat04.duration_p95'),
        ('IG01','trace.to_Tomcat04.row_count'),
        ('IG01','trace.self.row_count'),
        ('Tomcat04','trace.self.duration_p95'),
        ('Tomcat04','trace.self.row_count')
    ]
    start_local = tz.localize(pd.to_datetime('2021-03-23 01:30:00'))
    end_local   = tz.localize(pd.to_datetime('2021-03-23 02:00:00'))
    mask_window = (trace_df['dt'] >= start_local) & (trace_df['dt'] <= end_local)
    mask_group = trace_df.apply(lambda r: (r['cmdb_id'], r['trace_name']) in set(targets), axis=1)
    filtered = trace_df[mask_window & mask_group].copy()
    filtered['timestamp'] = filtered['dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    selected_rows = filtered[['timestamp','cmdb_id','trace_name','value']].sort_values('timestamp', ascending=True).head(200)

# Aggregate per group: count, min/max timestamp, max value
grp = selected_rows.groupby(['cmdb_id','trace_name'], as_index=False).agg(
    count_points = ('value','count'),
    min_time = ('timestamp','min'),
    max_time = ('timestamp','max'),
    max_value = ('value','max')
).sort_values('count_points', ascending=False)

total_rows = int(selected_rows.shape[0])
total_groups = int(grp.shape[0])
overall_min = selected_rows['timestamp'].min()
overall_max = selected_rows['timestamp'].max()

lines = []
lines.append(f"Filtered trace rows: {total_rows} rows for {total_groups} trace groups between {overall_min} and {overall_max} (UTC+8).")
lines.append("Per-group summary (count, time range, max_value) — up to 10 groups shown:")
for _, r in grp.head(10).iterrows():
    lines.append(f" - {r['cmdb_id']} | {r['trace_name']} — count={int(r['count_points'])}, range={r['min_time']} to {r['max_time']}, max_value={float(r['max_value']):.6g}")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

timestamp   cmdb_id                      trace_name     value
2     2021-03-23 01:30:00 CST      IG01            trace.self.row_count  106.0000
13    2021-03-23 01:30:00 CST      IG01  trace.to_Tomcat04.duration_p95    0.2022
14    2021-03-23 01:30:00 CST      IG01     trace.to_Tomcat04.row_count   13.0000
157   2021-03-23 01:30:00 CST  Tomcat04         trace.self.duration_p95    0.0230
158   2021-03-23 01:30:00 CST  Tomcat04            trace.self.row_count  187.0000
...                       ...       ...                             ...       ...
6373  2021-03-23 01:59:00 CST  Tomcat04         trace.self.duration_p95    0.0204
6224  2021-03-23 01:59:00 CST      IG01            trace.self.row_count   48.0000
6235  2021-03-23 01:59:00 CST      IG01  trace.to_Tomcat04.duration_p95    0.0582
6236  2021-03-23 01:59:00 CST      IG01     trace.to_Tomcat04.row_count    5.0000
6374  2021-03-23 01:59:00 CST  Tomcat04            trace.self.row_count   65.0000

[150 rows x 4 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz
import numpy as np

tz = pytz.timezone('Asia/Shanghai')

# Ensure trace_df is loaded (reuse if present)
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

# Ensure dt column exists and is timezone-aware Asia/Shanghai
if 'dt' in trace_df.columns:
    trace_df['dt'] = pd.to_datetime(trace_df['dt'])
    if trace_df['dt'].dt.tz is None:
        if 'timestamp' in trace_df.columns:
            trace_df['dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        else:
            trace_df['dt'] = trace_df['dt'].dt.tz_localize('UTC').dt.tz_convert(tz)
    else:
        trace_df['dt'] = trace_df['dt'].dt.tz_convert(tz)
else:
    trace_df['dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Target groups
targets = [
    ('IG01','trace.self.row_count'),
    ('IG01','trace.to_Tomcat04.row_count'),
    ('Tomcat04','trace.self.row_count')
]

# Compute full-series thresholds (global P5, P15, median) for the selected groups BEFORE window filtering
sel_mask_full = trace_df.apply(lambda r: (r['cmdb_id'], r['trace_name']) in set(targets), axis=1)
full_sel = trace_df[sel_mask_full].copy()

agg_full = full_sel.groupby(['cmdb_id','trace_name'], as_index=False).agg(
    p5 = ('value', lambda s: float(s.quantile(0.05))),
    p15 = ('value', lambda s: float(s.quantile(0.15))),
    median = ('value', lambda s: float(s.median()))
)

# Define window (inclusive) in UTC+8
start_local = tz.localize(pd.to_datetime('2021-03-23 01:30:00'))
end_local   = tz.localize(pd.to_datetime('2021-03-23 02:00:00'))

# Filter windowed data for the same groups
window_mask = (trace_df['dt'] >= start_local) & (trace_df['dt'] <= end_local) & sel_mask_full
trace_win = trace_df[window_mask].copy().sort_values(['cmdb_id','trace_name','dt'])

# Prepare results
rows = []
for _, g in agg_full.iterrows():
    cmdb = g['cmdb_id']
    name = g['trace_name']
    p5 = float(g['p5'])
    p15 = float(g['p15'])
    median_full = float(g['median'])

    grp = trace_win[(trace_win['cmdb_id']==cmdb) & (trace_win['trace_name']==name)].sort_values('dt').reset_index(drop=True)
    count_points_in_window = int(grp.shape[0])

    # earliest time value <= P15 and <= P5
    if count_points_in_window == 0:
        earliest_below_p15 = None
        earliest_below_p5 = None
        consecutive_runs = 0
        longest_run = 0
        min_val_in_window = None
    else:
        vals = grp['value'].astype(float)
        dts = grp['dt']

        mask_below_p15 = vals <= p15
        mask_below_p5 = vals <= p5

        if mask_below_p15.any():
            earliest_below_p15 = dts[mask_below_p15].iloc[0].tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
        else:
            earliest_below_p15 = None

        if mask_below_p5.any():
            earliest_below_p5 = dts[mask_below_p5].iloc[0].tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
        else:
            earliest_below_p5 = None

        # Compute consecutive runs for mask_below_p15 using time continuity (1 minute gaps)
        if mask_below_p15.any():
            anom_times = dts[mask_below_p15].reset_index(drop=True)
            # compute diffs in seconds
            diffs = anom_times.diff().dt.total_seconds().fillna(9999)
            # start new run when diff > 60
            run_id = (diffs > 60).cumsum()
            run_df = pd.DataFrame({'run_id': run_id, 'time': anom_times})
            run_lengths = run_df.groupby('run_id').size().values
            consecutive_runs = int(len(run_lengths))
            longest_run = int(int(run_lengths.max()))
        else:
            consecutive_runs = 0
            longest_run = 0

        min_val_in_window = float(vals.min()) if not vals.empty else None

    # max_drop_ratio_in_window = min_value_in_window / median_full (use min to represent drop)
    if (min_val_in_window is None) or (median_full == 0):
        max_drop_ratio = None
    else:
        max_drop_ratio = float(min_val_in_window / median_full)

    rows.append({
        'cmdb_id': cmdb,
        'trace_name': name,
        'p5': p5,
        'p15': p15,
        'count_points_in_window': count_points_in_window,
        'earliest_time_value_below_P15': earliest_below_p15,
        'earliest_time_value_below_P5': earliest_below_p5,
        'consecutive_below_P15_runs': consecutive_runs,
        'longest_run_below_P15': longest_run,
        'max_drop_ratio_in_window': max_drop_ratio
    })

out_df = pd.DataFrame(rows, columns=[
    'cmdb_id','trace_name','p5','p15','count_points_in_window',
    'earliest_time_value_below_P15','earliest_time_value_below_P5',
    'consecutive_below_P15_runs','longest_run_below_P15','max_drop_ratio_in_window'
])

# Format numeric columns for compact display precision
out_df['p5'] = out_df['p5'].map(lambda x: float(f"{x:.6g}"))
out_df['p15'] = out_df['p15'].map(lambda x: float(f"{x:.6g}"))
out_df['max_drop_ratio_in_window'] = out_df['max_drop_ratio_in_window'].map(lambda x: (float(f"{x:.6g}") if x is not None else None))

# Return compact table (3 rows)
out_df.head(3)

```
Out[8]:
```
```python
# Summarize the drop-detection results in plain English using the existing `out_df`.
# The summary is returned as a single string variable for compact display.

try:
    out_df
except NameError:
    raise RuntimeError("Expected `out_df` not found in the environment. Re-run prior analysis to produce thresholds and window results.")

lines = []
lines.append("Summary of lower-tail (drop) detection for selected trace groups (window 2021-03-23 01:30–02:00 UTC+8):")

for _, r in out_df.iterrows():
    cmdb = r['cmdb_id']
    name = r['trace_name']
    p5 = r['p5']
    p15 = r['p15']
    cnt = int(r['count_points_in_window'])
    t_p15 = r['earliest_time_value_below_P15'] or "None"
    t_p5 = r['earliest_time_value_below_P5'] or "None"
    runs = int(r['consecutive_below_P15_runs'])
    longest = int(r['longest_run_below_P15'])
    ratio = r['max_drop_ratio_in_window']
    ratio_str = f"{ratio:.3f}" if ratio is not None else "N/A"
    drop_pct = f"{(1-ratio)*100:.1f}%" if (ratio is not None) else "N/A"

    lines.append(
        f"- {cmdb} / {name}: full-series P5={p5}, P15={p15}. "
        f"In window: {cnt} points; earliest <=P15 at {t_p15}; earliest <=P5 at {t_p5}. "
        f"Consecutive <=P15 runs={runs}, longest run={longest} minute(s). "
        f"Max drop ratio (min/median)={ratio_str} (~{drop_pct} drop from median)."
    )

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                   trace_name    p5    p15  count_points_in_window earliest_time_value_below_P15 earliest_time_value_below_P5  consecutive_below_P15_runs  longest_run_below_P15  max_drop_ratio_in_window
0      IG01         trace.self.row_count  34.0  39.40                      30       2021-03-23 01:45:00 CST      2021-03-23 01:45:00 CST                           3                      2                  0.295082
1      IG01  trace.to_Tomcat04.row_count   4.0   5.35                      30       2021-03-23 01:45:00 CST      2021-03-23 01:50:00 CST                           5                      1                  0.500000
2  Tomcat04         trace.self.row_count  57.3  66.00                      30       2021-03-23 01:44:00 CST      2021-03-23 01:55:00 CST                           5                      3                  0.327731```
```